# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [1]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel

simPrefix = os.getcwd()+'\\simdata'

## Data Input

In [2]:
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [3]:
print(x.shape)
print(y.keys())
# each station has 2 tanks, each tank has 2 DOMs (high/log gain)
# each tank measures charge and time
# each station gives 2 charges and 2 times, 4 total pieces of data per station
# stations arranged in 10x10 square lattice, 2 corners of square unused
# charge measured in VEM, vertical equivalent muon

# 'dir' is true direction, rest of dir are reconstruted by simulations
# 'plane_dir' assumes shower is flat plane
# 'laputop_dir' performs likelihood analysis
# 'small_dir' compromises between plane and laputop

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [4]:
# 85/15 split for training/validation
energy = y['energy']
comp = y['comp']
theta, phi = y['dir'].transpose()
nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)

## Model Training

### Alpha Model
- Input: no charge merge, no time layers included, normalized data, combined with cosine of zenith angle
- Layers: Two convolutional layers for charge, then combined with zenith
- Output: Energy

In [5]:
# Name for model
key = 'q1q2cosZ'
numepochs = 6
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':None}

In [9]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(1), name="charge")

dense1_layer = layers.Dense(64)(charge_input)
dense2_layer = layers.Dense(16)(dense1_layer)
output = layers.Dense(1)(dense2_layer)

model = models.Model(inputs=[charge_input],outputs=output,name=key)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

## Old model used for reference
#model = Sequential(name=nameModel(prep, 'test'))  # Automatic naming for flexible assessment later
## Add model layers
#model.add(Conv2D(64, kernel_size=3, activation='relu', input_shape=(10,10,2)))
#model.add(Conv2D(32, kernel_size=3, activation='relu'))
#model.add(Flatten())
#model.add(Dense(1)) # No activation function for last layer of regression model

## Compile model
#model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [14]:
# Establish arrays to be trained on
x_i = dataPrep(x, y, **prep)
temp_y = energy
x_i1 = np.sum(np.sum(np.sum(x_i,axis=3),axis=2),axis=1)
x_i1.shape

(549773,)

In [11]:
model.summary()

Model: "q1q2cosZ"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
charge (InputLayer)          [(None, 1)]               0         
_________________________________________________________________
dense (Dense)                (None, 64)                128       
_________________________________________________________________
dense_1 (Dense)              (None, 16)                1040      
_________________________________________________________________
dense_2 (Dense)              (None, 1)                 17        
Total params: 1,185
Trainable params: 1,185
Non-trainable params: 0
_________________________________________________________________


In [9]:
keras.utils.plot_model(model,"model.png")

('You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) ', 'for plot_model/model_to_dot to work.')


In [15]:
# Train
history = model.fit(
    {"charge":x_i1}, temp_y, epochs=numepochs,validation_split=0.15)

Epoch 1/6
14604/14604 [==============================] - 9s 574us/step - loss: 0.5140 - mae: 0.5077 - mse: 0.5140 - val_loss: 0.3117 - val_mae: 0.4550 - val_mse: 0.3117
Epoch 2/6
14604/14604 [==============================] - 8s 568us/step - loss: 0.3249 - mae: 0.4671 - mse: 0.3249 - val_loss: 0.2913 - val_mae: 0.4439 - val_mse: 0.2913
Epoch 3/6
14604/14604 [==============================] - 8s 569us/step - loss: 0.3232 - mae: 0.4658 - mse: 0.3232 - val_loss: 0.3173 - val_mae: 0.4493 - val_mse: 0.3173
Epoch 4/6
14604/14604 [==============================] - 9s 601us/step - loss: 0.3225 - mae: 0.4652 - mse: 0.3225 - val_loss: 0.2881 - val_mae: 0.4413 - val_mse: 0.2881
Epoch 5/6
14604/14604 [==============================] - 9s 631us/step - loss: 0.3214 - mae: 0.4645 - mse: 0.3214 - val_loss: 0.2897 - val_mae: 0.4412 - val_mse: 0.2897
Epoch 6/6
14604/14604 [==============================] - 9s 633us/step - loss: 0.3208 - mae: 0.4641 - mse: 0.3208 - val_loss: 0.2943 - val_mae: 0.4393 - va

In [11]:
# Save model to file for easy loading
## WHERE ARE YOU SAVING TO?
model.save('model_%s.h5' % key)
f = open("results.txt", "a")
now = datetime.now()
f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    now.strftime("%m/%d/%Y %H:%M:%S"),
    key,
    numepochs,
    history.history['loss'][numepochs-1],
    history.history['val_loss'][numepochs-1]
))
f.close()

## Your task

- **Create your own model**
- Replace the model here w/ *simplified* form of Brandon's model (focus: including zenith)
- change the zenith input to cosine(zenith) input